# UAV Sidewalk — Swin-Large, keyframe 8, work 960
Cấu hình đã chọn: Mask2Former Swin-Large Mapillary, PyTorch eager FP16 trên T4.
Model/postprocess 768, tracking 640, work/output 960, smoothing 0.3, frame_step=1.
Chỉ còn một luồng chạy: thiết lập → chọn video → phân tích → xem → tải kết quả.
Không dùng PIDNet, SegFormer, torch.compile, TensorRT hoặc AI/encoder bất đồng bộ.
**13,62 FPS là kết quả của một lượt thử trước, không phải tốc độ được đảm bảo.**
Ở đầu vào 15 FPS, keyframe 8 tương ứng khoảng 0,53 giây giữa các nhận diện AI
khi không có fallback. Giữ kiểm tra tracking và chạy AI sớm khi không đáng tin.
Chưa có kiểm định nhãn chuẩn hoặc bảo đảm realtime cho mọi cảnh drone.
Bật Runtime → Change runtime type → T4 GPU trước khi chạy.


## 1. Kiểm tra GPU và Môi trường

In [ ]:
import subprocess
import sys
import torch
subprocess.run(['nvidia-smi'], check=True)
print('Python:', sys.version)
print('PyTorch:', torch.__version__, '| CUDA build:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.')
print('GPU:', torch.cuda.get_device_name(0))


## 2. Kết nối Google Drive và cài đúng phiên bản

Upload `UAV_code_optimized.zip` mới; notebook ưu tiên gói này, không dùng cache Python cũ.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

drive.mount('/content/drive')
ZIP_PATH = ''  # Điền đường dẫn ZIP cụ thể nếu có nhiều bản khác nhau.
candidates = [
    Path('/content/UAV_code_optimized.zip'),
    Path('/content/drive/MyDrive/UAV/UAV_code_optimized.zip'),
    Path('/content/drive/MyDrive/UAV_code_optimized.zip'),
]
available = [path for path in candidates if path.is_file()]
if ZIP_PATH:
    source_zip = Path(ZIP_PATH).expanduser()
elif not available:
    raise FileNotFoundError('Upload UAV_code_optimized.zip mới vào /content hoặc Drive/MyDrive/UAV.')
else:
    hashes = {hashlib.sha256(path.read_bytes()).hexdigest() for path in available}
    if len(hashes) > 1:
        raise RuntimeError('Có nhiều ZIP khác nhau. Điền ZIP_PATH rõ ràng: ' + str(available))
    source_zip = available[0]
print('ZIP đang dùng:', source_zip)
root = Path('/content/uav_sidewalk_selected').resolve()
root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(source_zip) as archive:
    if len(archive.namelist()) != len(set(archive.namelist())):
        raise ValueError('ZIP chứa tên trùng lặp')
    manifest = json.loads(archive.read('build_manifest.json'))
    for name, digest in manifest['sha256'].items():
        target = (root / name).resolve()
        if not target.is_relative_to(root):
            raise ValueError('Unsafe archive path: ' + name)
        if hashlib.sha256(archive.read(name)).hexdigest() != digest:
            raise ValueError('Sai checksum: ' + name)
    for member in archive.infolist():
        if not (root / member.filename).resolve().is_relative_to(root):
            raise ValueError('Unsafe archive path: ' + member.filename)
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('ZIP không được chứa symlink')
    if manifest.get('code_layout') != 'functional-v1':
        raise RuntimeError('ZIP dùng cấu trúc code cũ. Upload lại UAV_code_optimized.zip mới.')
    if manifest.get('selected_profile') != 'swin-large-k8-work960':
        raise RuntimeError('ZIP không đúng bản đã chốt Swin-Large keyframe 8/work 960. Upload ZIP mới.')
    archive.extractall(root)
for name, digest in manifest['sha256'].items():
    if hashlib.sha256((root / name).read_bytes()).hexdigest() != digest:
        raise RuntimeError('Code sau giải nén không khớp: ' + name)
os.chdir(root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-c',
    "import torch,transformers,cv2; print('torch',torch.__version__,'transformers',transformers.__version__,'opencv',cv2.__version__); assert torch.cuda.is_available(), 'Enable the Colab GPU runtime'"], check=True)
print('Đã xác minh checksum bản Swin-Large keyframe 8/work 960.')

def run_logged(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Command:', command, flush=True)
    print('Log:', log_path, flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    log.write(line)
                    log.flush()
                    print(line, end='', flush=True)
                return process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise


## 3. Chọn video UAV cần phân tích
- Bạn có thể tải video từ máy tính lên bằng ô code bên dưới.
- Hoặc nếu video đã có sẵn trên Google Drive, bạn chỉ cần gán đường dẫn vào biến `video_path`.

In [ ]:
from google.colab import files
from pathlib import Path

INPUT_VIDEO = ''  # Điền đường dẫn Drive hoặc /content nếu đã có video.
DRIVE_VIDEO_INDEX = None  # Chọn chỉ số trong danh sách; None = upload từ máy.
suffixes = {'.mp4', '.mov', '.avi', '.mkv', '.webm', '.m4v'}
drive_folder = Path('/content/drive/MyDrive/UAV')
available_videos = sorted(path for path in drive_folder.glob('*') if path.suffix.lower() in suffixes)
for index, path in enumerate(available_videos):
    print(index, path.name)
if INPUT_VIDEO:
    video_path = str(Path(INPUT_VIDEO).expanduser())
elif DRIVE_VIDEO_INDEX is not None:
    if not 0 <= DRIVE_VIDEO_INDEX < len(available_videos):
        raise ValueError('DRIVE_VIDEO_INDEX không hợp lệ')
    video_path = str(available_videos[DRIVE_VIDEO_INDEX])
else:
    print('Chọn đúng một video từ máy tính:')
    uploaded = files.upload()
    selected = [name for name in uploaded if Path(name).suffix.lower() in suffixes]
    if len(selected) != 1:
        raise ValueError('Cần chọn đúng một video được hỗ trợ')
    video_path = selected[0]
print('Video đã chọn:', video_path)


### Sao chép video đã chọn vào ổ cục bộ Colab
Chạy ô này trước khi benchmark để loại trừ độ trễ đọc từ Drive.


In [ ]:
from pathlib import Path
import shutil
import cv2

video_source = Path(video_path).expanduser()
if not video_source.is_file():
    raise FileNotFoundError(str(video_source))
input_dir = Path("/content/uav_inputs")
input_dir.mkdir(exist_ok=True)
local_video = input_dir / video_source.name
if video_source.resolve() != local_video.resolve():
    print("Copying video to the Colab VM; this is setup time, not inference time.")
    shutil.copy2(video_source, local_video)
video_path = str(local_video)
capture = cv2.VideoCapture(video_path)
if not capture.isOpened():
    raise RuntimeError(f"Cannot open video: {video_path}")
print("Local input:", video_path)
print("Frames:", capture.get(cv2.CAP_PROP_FRAME_COUNT), "FPS:", capture.get(cv2.CAP_PROP_FPS))
print("Source dimensions:", capture.get(cv2.CAP_PROP_FRAME_WIDTH), capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
capture.release()


## 4. Phân tích video bằng cấu hình đã chọn
Chạy đủ frame, không sửa FPS phát để tạo số đo tốc độ.
Mỗi lượt tạo thư mục output riêng; log và profile lưu cùng kết quả.
MAX_FRAMES=0 xử lý toàn bộ video; giá trị khác 0 dùng để kiểm tra ngắn.


In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

MODEL_CONFIG = "configs/gpu_t4.json"
KEYFRAME_INTERVAL = 8
WORK_MAX_SIDE = 960
MAX_FRAMES = 0
from uuid import uuid4
OUTPUT_ROOT = str(Path("/content/outputs_sidewalk") / uuid4().hex[:10])

command = [
    sys.executable, "-u", "-m", "road_sidewalk_seg",
    "--config", MODEL_CONFIG, "--input", video_path,
    "--output", OUTPUT_ROOT,
    "--keyframe-interval", str(KEYFRAME_INTERVAL),
    "--video-frame-step", "1", "--video-max-frames", str(MAX_FRAMES),
    "--video-temporal-smoothing", "0.3",
    "--video-max-side", "768",
    "--video-work-max-side", str(WORK_MAX_SIDE),
    "--video-tracking-max-side", "640",
    "--video-log-interval", "2",
    "--video-postprocess-max-side", "768",
    "--video-cv-threads", "1", "--video-target-fps", "15",
    "--video-mask-sample-step", "30",
    "--headless", "--save-video", "--fail-fast",
]
print("Command:", command)
run_started = time.perf_counter()
returncode = run_logged(command, Path(OUTPUT_ROOT) / 'run.log')
if returncode:
    raise RuntimeError(f'Phân tích thất bại (exit {returncode}); xem run.log')
print("CLI wall time including Python startup:", round(time.perf_counter() - run_started, 2), "s")
profile_path = Path(OUTPUT_ROOT) / Path(video_path).stem / "video_profile.json"
profile = json.loads(profile_path.read_text())
print("Actual runtime:", profile["runtime"])
print("Actual processed FPS:", profile["processed_fps"])
print("Speed relative to source:", profile["speed_vs_source"], "(>= 1 means file throughput reaches source rate)")
print("AI / warp / fallback counts:", profile["counts"])
print("Stages:", profile["stage_seconds"])
print("AI stages:", profile["ai_stage_seconds"])
print("Fallback reasons:", profile["fallback_reasons"])
print("Profile:", profile_path)

print("15 FPS throughput target met:", profile["throughput_target_met"])
print("AI refreshes / wall second:", profile["ai_updates_per_wall_second"])
print("Quality verified:", profile["quality_verified"], "(requires separate mask review/labels)")
if not profile["throughput_target_met"]:
    print("NOT MET: send video_profile.json; do not use playback FPS as processing FPS.")



## 5. Xem video kết quả

Bước chuyển H.264 và tải video vào trình duyệt được đo riêng, không tính vào tốc độ suy luận.


In [ ]:
from IPython.display import Video, display
from pathlib import Path
import subprocess
import time

raw_result = Path(OUTPUT_ROOT) / Path(video_path).stem / "preview.mp4"
web_result = raw_result.with_name("preview_web.mp4")
if not raw_result.exists():
    raise FileNotFoundError(str(raw_result))
conversion_started = time.perf_counter()
subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error", "-i", str(raw_result),
    "-c:v", "libx264", "-preset", "veryfast", "-crf", "20",
    "-pix_fmt", "yuv420p", str(web_result),
], check=True)
print("Browser conversion time:", round(time.perf_counter() - conversion_started, 2), "s")
display(Video(str(web_result), embed=True, width=768))



## 6. Tải Video kết quả về máy tính

In [ ]:
from google.colab import files
download_target = web_result if web_result.exists() else raw_result
if download_target.exists():
    print(f"Đang tải {download_target.name} về máy...")
    files.download(str(download_target))
else:
    print("Chưa có file để tải về!")
